In [33]:
import pandas as pd
from pandas.api.typing import NAType

import numpy as np

In [34]:
# Load dataset
file_name = '../../data/2018-2025/ncsl_ai_db_2018_2025.csv'
df = pd.read_csv(file_name)

df = df.fillna("")

In [35]:
# Rename column names for easier access
df = df.rename(columns={
    "State": "state",
    "Bill Number": "bill_number",
    "Bill Title": "title",
    "Bill Summary": "summary",
    "Bill Status": "status",
    "Category": "category",
    "URL": "url",
    "Page Year": "page_year",
})

# Remove rows with no bill, no legislative session, or data
is_removed = (df.bill_number == "") | df.bill_number.str.match(r"No\b|Not\b")
df = df[~is_removed]

In [36]:
# Normalize, remove dashes and trailing whitespaces from text columns
for column in df.columns[:-2]:
    df[column] = (df[column].str.normalize("NFKC")
                  .str.replace("[\u2010-\u2015\u2212]", "-", regex=True)
                  .str.replace(r"\s+", " ", regex=True)
                  .str.strip())

In [37]:
# Fix spelling mistake in 'status' column
df.status = df.status.apply(
    lambda s: s if s != 'To governor' else 'To Governor')

In [38]:
# Fix special case scraping error for 'bill_number'
df.loc[df.bill_number == "B24-558", "bill_number"] = "B 558"

In [39]:
import base64
import json
import re

def url_bill_id(u) -> str | None:
    """Extract bill ID from each bill's url source"""
    if not isinstance(u, str) or not u:
        return None

    # BILL SOURCE --> StateNet
    m = re.search(r"ID:bill:([A-Za-z0-9]+)", u)
    if m:
        return m.group(1)

    # BILL SOURCE --> California leginfo site, ex. CA202120220AB13
    m = re.search(r"bill_id=(\d{8}\d?[A-Z]+\d+)", u)
    if m:
        return "CA" + m.group(1)

    # BILL SOURCE --> Maine legislature site, ex. ME130HP1199
    m = re.search(r"paper=([A-Z]+\d+).*snum=(\d+)", u)
    if m:
        return "ME" + m.group(2) + m.group(1)

    # BILL SOURCE --> LexisNexis website (comes in base64 JSON)
    if "lexisnexis.com/opentext/" in u:
        token = u.split("/opentext/")[1].split(".")[0]
        try:
            return json.loads(base64.urlsafe_b64decode(token + "=" * (-len(token) % 4)))["id"]
        except Exception:
            return None

    return None


df["bill_id"] = df.url.map(url_bill_id)

# Bills without a usable link get a fallback id instead of NaN
df["bill_id"] = df.bill_id.fillna(df.state + "_" + df.page_year.astype(str) + "_" + df.bill_number.str.replace(" ", ""))

assert df.bill_id.notna().all()

In [40]:
df["is_special_session"] = df.bill_number.str.contains("session", case=False).groupby(df.bill_id).transform("any")

In [41]:
# Sort so "last" always means the latest page, and flag special sessions BEFORE the duplicate listings are dropped
df = df.sort_values(["bill_id", "page_year"])
df["is_special_session"] = df.bill_number.str.contains("session", case=False).groupby(df.bill_id).transform("any")

# Every listing of every bill (status can change between pages) - exported at end
history = df.sort_values(["bill_id", "page_year"]).copy()

In [42]:
# Fill title/category gaps from the latest earlier listing of the same bill that has a value
for c in ["title", "category"]:
    latest = (df.assign(**{c: df[c].replace("", np.nan)})
                .dropna(subset=[c]).groupby("bill_id")[c].last())
    df[c] = df[c].replace("", np.nan).fillna(df.bill_id.map(latest))

# Only keep the final/latest bill listing/status
df = df.drop_duplicates("bill_id", keep="last")

In [43]:
import us

state_mapping = us.states.mapping('name', 'abbr')
state_mapping.pop("Virgin Islands")
state_mapping['U.S. Virgin Islands'] = 'VI'

df.state = df.state.map(state_mapping)
history["state"] = history.state.map(state_mapping)

In [44]:
# Standardized categories retrieved from NCSL website (some have been renamed over the years)
# ex. 'Education Use' and 'Education/Training'
categories = ['Appropriations', 'Audit', 'Child Pornography', 'Criminal Use', 'Cybersecurity', 'Education Use', 'Education/Training',
              'Effect on Labor/Employment', 'Elections', 'Energy', 'Government Use', 'Health Use', 'Housing', 'Impact Assessment',
              'Judicial Use', 'Notification', 'Oversight/Governance', 'Personhood', 'Private Right of Action', 'Private Sector',
              'Provenance', 'Responsible Use', 'Studies', 'Taxes']


def clean_categories(s) -> str | NAType:
    """Each category is formatted as a list of labels separated by a semi-colon"""
    if pd.isna(s):
        return pd.NA
    found = [c for c in categories if c.lower() in s.lower()]
    return ";".join(found) if found else pd.NA

# Make category into a ";"-separated list of the categories above
# Also, check every piece of raw category text must match one of the categories above
cat_text = df.category.str.replace(";", ",").str.split(",").explode().str.strip()
cat_text = cat_text[cat_text.notna() & (cat_text != "")]
unmatched_categories = cat_text[~cat_text.str.lower().map(lambda t: any(c.lower() in t for c in categories))]
assert unmatched_categories.empty, f"Unrecognized category text: {unmatched_categories.unique()}"

df["category"] = df.category.map(clean_categories)

print("Remaining bills without categories: ", df.category.isna().sum())

Remaining bills without categories:  76


In [45]:
# Replace curly quotes within 'title' and 'summary' text columns
quotes = str.maketrans({"\u2018": "'", "\u2019": "'",
                       "\u201c": '"', "\u201d": '"'})

for c in ["title", "summary"]:
    df[c] = df[c].map(lambda s: s.translate(quotes)
                      if isinstance(s, str) else s)

In [46]:
# Final checks - each bill is unique and no null values
assert df.bill_id.is_unique
assert df.state.notna().all() and df.bill_number.notna().all()
assert df.bill_id.notna().all()
assert set(df.category.dropna().str.split(";").explode()) <= set(categories)

# assert (df.isnull().sum().sum() == 0)
print(df.isnull().sum()[lambda s: s > 0])

title       41
category    76
dtype: int64


In [47]:
df = df.replace("", np.nan)

# Export - Bill status history
history[["bill_id", "state", "page_year", "status"]].to_csv(
    "../../data/2018-2025/bill_status_history_2018_2025.csv", index=False)

In [48]:
# Reorder column names before export - dropping 'url' in the process
column_order = ["bill_id", "bill_number", "state", "status", "category",
                 "title", "summary", "is_special_session", "page_year"]
df = df[column_order]

# Export - Entinre cleaned dataset
file_name = '../../data/2018-2025/ncsl_ai_db_2018_2025_clean.csv'
df.to_csv(file_name, index=False)